# AVG Training v5 — CNN custom lalu SVM


## 1. Konfigurasi dan direktori keluaran v5


In [1]:
from pathlib import Path
import gc, hashlib, json, os, time
import joblib
import numpy as np
import tensorflow as tf
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.svm import SVC

SEED = 30092026
RUN_ID = "cifar10_7fitur_v5"
VARIANT = "avg"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / VARIANT
for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
if tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Varian:", VARIANT, "| GPU:", tf.config.list_physical_devices("GPU"))


I0000 00:00:1791576421.096796   71613 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791576421.133203   71613 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1791576422.078005   71613 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Varian: avg | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Muat CIFAR-10 dan buat split tetap


In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
assert x_all.shape == (50000, 32, 32, 3)
idx_train, idx_val = train_test_split(
    np.arange(50000), test_size=0.2, stratify=y_all, random_state=SEED
)
idx_train, idx_val = np.sort(idx_train), np.sort(idx_val)
split_hash = hashlib.sha256(idx_train.tobytes() + idx_val.tobytes()).hexdigest()
assert len(idx_train) == 40000 and len(idx_val) == 10000
screen_train, _ = train_test_split(
    idx_train, train_size=20000, stratify=y_all[idx_train], random_state=SEED
)
screen_val, _ = train_test_split(
    np.setdiff1d(idx_train, screen_train),
    train_size=5000,
    stratify=y_all[np.setdiff1d(idx_train, screen_train)],
    random_state=SEED,
)
screen_train, screen_val = np.sort(screen_train), np.sort(screen_val)
assert len(np.intersect1d(screen_train, screen_val)) == 0
print("Split:", split_hash, "| CNN screening:", len(screen_train), len(screen_val))


Split: 16004a1ac8a89ac92ba6280367a7fb7b284c3643abb1c9d10409d6739ef31305 | CNN screening: 20000 5000


## 3. Konversi input RGB/AVG/NTSC


In [3]:
def prepare_images(x, variant):
    rgb = np.asarray(x, dtype=np.float32) / np.float32(255.0)
    if variant == "rgb":
        return rgb
    if variant == "avg":
        return np.mean(rgb, axis=-1, keepdims=True, dtype=np.float32)
    if variant == "ntsc":
        weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
        return np.sum(rgb * weights, axis=-1, keepdims=True, dtype=np.float32)
    raise ValueError("Varian tidak dikenal: " + variant)


## 4. Augmentasi batch saat training saja


In [4]:
def make_training_dataset(images, labels, policy, batch_size, seed):
    images = prepare_images(images, VARIANT)
    labels = tf.one_hot(np.asarray(labels, dtype=np.int32), 10, dtype=tf.float32)
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    dataset = dataset.shuffle(
        len(images), seed=seed, reshuffle_each_iteration=True
    ).batch(batch_size)

    def augment_batch(x, y):
        count = tf.shape(x)[0]
        channels = tf.shape(x)[-1]
        if policy == "cutout":
            cy = tf.random.uniform((), 0, 32, dtype=tf.int32)
            cx = tf.random.uniform((), 0, 32, dtype=tf.int32)
            rows = tf.range(32)[:, None]
            cols = tf.range(32)[None, :]
            mask = tf.cast((tf.abs(rows - cy) < 4) & (tf.abs(cols - cx) < 4), x.dtype)
            return x * (1.0 - mask[None, :, :, None]), y
        perm = tf.random.shuffle(tf.range(count))
        other_x = tf.gather(x, perm)
        other_y = tf.gather(y, perm)
        alpha = tf.constant(0.2 if policy == "mixup" else 1.0, tf.float32)
        a = tf.random.gamma((), alpha)
        b = tf.random.gamma((), alpha)
        lam = a / tf.maximum(a + b, 1e-7)
        if policy == "mixup":
            return lam * x + (1.0 - lam) * other_x, lam * y + (1.0 - lam) * other_y
        if policy == "cutmix":
            width = tf.cast(tf.round(tf.sqrt(1.0 - lam) * 32.0), tf.int32)
            cy = tf.random.uniform((), 0, 32, dtype=tf.int32)
            cx = tf.random.uniform((), 0, 32, dtype=tf.int32)
            rows = tf.range(32)[:, None]
            cols = tf.range(32)[None, :]
            mask = tf.cast(
                (tf.abs(rows - cy) < width // 2) & (tf.abs(cols - cx) < width // 2),
                x.dtype,
            )
            mask = mask[None, :, :, None]
            area = tf.reduce_mean(mask)
            return x * (1.0 - mask) + other_x * mask, (1.0 - area) * y + area * other_y
        raise ValueError("Kebijakan augmentasi tidak dikenal: " + policy)

    return dataset.map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(
        tf.data.AUTOTUNE
    )


## 5. Definisi CNN residual custom dan optimizer


In [5]:
def residual_block(x, filters, stride, name):
    shortcut = x
    x = tf.keras.layers.Conv2D(
        filters, 3, strides=stride, padding="same", use_bias=False, name=name + "_conv1"
    )(x)
    x = tf.keras.layers.BatchNormalization(name=name + "_bn1")(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Conv2D(
        filters, 3, padding="same", use_bias=False, name=name + "_conv2"
    )(x)
    x = tf.keras.layers.BatchNormalization(name=name + "_bn2")(x)
    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = tf.keras.layers.Conv2D(
            filters, 1, strides=stride, use_bias=False, name=name + "_skipconv"
        )(shortcut)
        shortcut = tf.keras.layers.BatchNormalization(name=name + "_skipbn")(shortcut)
    return tf.keras.layers.Activation("relu", name=name + "_relu")(
        tf.keras.layers.Add(name=name + "_add")([x, shortcut])
    )


def build_cnn(variant):
    channels = 3 if variant == "rgb" else 1
    inputs = tf.keras.Input((32, 32, channels), name=variant + "_input")
    x = tf.keras.layers.ZeroPadding2D(4)(inputs)
    x = tf.keras.layers.RandomCrop(32, 32, seed=SEED)(x)
    x = tf.keras.layers.RandomFlip("horizontal", seed=SEED)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    for filters, stride in ((64, 1), (128, 2), (256, 2), (512, 2)):
        x = residual_block(x, filters, stride, "stage" + str(filters) + "_block1")
        x = residual_block(x, filters, 1, "stage" + str(filters) + "_block2")
    embedding = tf.keras.layers.GlobalAveragePooling2D(
        dtype="float32", name="embedding"
    )(x)
    x = tf.keras.layers.Dropout(0.3)(embedding)
    outputs = tf.keras.layers.Dense(10, activation="softmax", dtype="float32")(x)
    return tf.keras.Model(inputs, outputs, name="custom_" + variant + "_residual_v5")


def compile_cnn(model, lr, epochs, steps):
    schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=lr, decay_steps=max(1, epochs * steps), alpha=0.01
    )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(schedule, weight_decay=5e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return model


## 6. Estimator scikit-learn untuk pencarian CNN


In [6]:
class CNNGridEstimator(ClassifierMixin, BaseEstimator):
    def __init__(
        self, variant="rgb", policy="cutout", lr=0.0003, epochs=25, batch_size=32
    ):
        self.variant, self.policy, self.lr = variant, policy, lr
        self.epochs, self.batch_size = epochs, batch_size

    def fit(self, X, y):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        self.model_ = compile_cnn(
            build_cnn(self.variant),
            self.lr,
            self.epochs,
            int(np.ceil(len(X) / self.batch_size)),
        )
        dataset = make_training_dataset(X, y, self.policy, self.batch_size, SEED)
        self.model_.fit(dataset, epochs=self.epochs, verbose=1)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        x = prepare_images(X, self.variant)
        scores = self.model_.predict(x, batch_size=self.batch_size, verbose=1)
        return np.argmax(scores, axis=1)


## 7. GridSearchCV CNN dengan ledger per kandidat


In [7]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    os.replace(temporary, path)


ledger_path = OUT / "laporan" / "cnn_grid_ledger.json"
data_digest = hashlib.sha256()
data_digest.update(x_all.tobytes())
data_digest.update(y_all.tobytes())
identity = {
    "run_id": RUN_ID,
    "variant": VARIANT,
    "seed": SEED,
    "split_hash": split_hash,
    "data_sha256": data_digest.hexdigest(),
    "screen_train_sha256": hashlib.sha256(screen_train.tobytes()).hexdigest(),
    "screen_val_sha256": hashlib.sha256(screen_val.tobytes()).hexdigest(),
    "architecture": "residual-64-128-256-512-two-blocks-gap512",
    "epochs": 25,
    "batch_size": 32,
    "weight_decay": 5e-4,
    "label_smoothing": 0.05,
}
identity_sha = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
if ledger_path.exists():
    ledger = json.loads(ledger_path.read_text(encoding="utf-8"))
    if ledger["identity_sha256"] != identity_sha:
        raise ValueError("Ledger CNN berbeda dari data/konfigurasi v5")
else:
    ledger = {"identity_sha256": identity_sha, "identity": identity, "results": {}}
    atomic_json(ledger_path, ledger)
if VARIANT == "rgb":
    policies = ("cutout", "mixup", "cutmix")
else:
    rgb_ledger_path = (
        ROOT / "outputs" / RUN_ID / "rgb" / "laporan" / "cnn_grid_ledger.json"
    )
    if not rgb_ledger_path.exists():
        raise FileNotFoundError("Jalankan RGB Training v5 lebih dahulu")
    rgb_ledger = json.loads(rgb_ledger_path.read_text(encoding="utf-8"))
    ranked = sorted(
        rgb_ledger["results"].values(), key=lambda item: item["accuracy"], reverse=True
    )
    policies = tuple(dict.fromkeys(item["policy"] for item in ranked))[:2]
    if len(policies) != 2:
        raise ValueError("Hasil grid RGB belum mencakup dua kebijakan augmentasi")
search_indices = np.r_[screen_train, screen_val]
fold = PredefinedSplit(
    np.r_[
        np.full(len(screen_train), -1, dtype=np.int8),
        np.zeros(len(screen_val), dtype=np.int8),
    ]
)
for policy in policies:
    for lr in (0.0003, 0.001):
        key = f"{policy}|{lr:.4g}"
        if key in ledger["results"]:
            print("Lewati kandidat valid:", key, flush=True)
            continue
        estimator = CNNGridEstimator(variant=VARIANT)
        grid = GridSearchCV(
            estimator,
            {"policy": [policy], "lr": [lr]},
            scoring="accuracy",
            cv=fold,
            refit=False,
            n_jobs=1,
            verbose=1,
            error_score="raise",
        )
        began = time.perf_counter()
        grid.fit(x_all[search_indices], y_all[search_indices])
        ledger["results"][key] = {
            "policy": policy,
            "lr": lr,
            "accuracy": float(grid.best_score_),
            "seconds": time.perf_counter() - began,
            "n_train": len(screen_train),
            "n_val": len(screen_val),
        }
        atomic_json(ledger_path, ledger)
        print("CNN GridSearchCV:", key, ledger["results"][key], flush=True)
        mean_seconds = np.mean([item["seconds"] for item in ledger["results"].values()])
        remaining = 2 * len(policies) - len(ledger["results"])
        print(
            "Perkiraan sisa screening CNN (menit):",
            round(remaining * mean_seconds / 60, 1),
            flush=True,
        )
        del grid
        tf.keras.backend.clear_session()
        gc.collect()
best = max(ledger["results"].values(), key=lambda item: (item["accuracy"], -item["lr"]))
print("Kandidat CNN terpilih:", best)


Fitting 1 folds for each of 1 candidates, totalling 1 fits


I0000 00:00:1791576425.365339   71613 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3672 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


/home/saga/anaconda3/envs/envGPU/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


I0000 00:00:1791576434.930956   71686 cuda_dnn.cc:461] Loaded cuDNN version 92600


Epoch 1/25 — 625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 77ms/step - accuracy: 0.3097 - loss: 2.0744


...

Epoch 25/25 — 625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 84ms/step - accuracy: 0.9404 - loss: 0.4595


CNN GridSearchCV: cutout|0.001 {'policy': 'cutout', 'lr': 0.001, 'accuracy': 0.8658, 'seconds': 1314.4581629629974, 'n_train': 20000, 'n_val': 5000}


Perkiraan sisa screening CNN (menit): 0.0


Kandidat CNN terpilih: {'policy': 'cutout', 'lr': 0.001, 'accuracy': 0.8658, 'seconds': 1314.4581629629974, 'n_train': 20000, 'n_val': 5000}


157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step


## 8. Training CNN terpilih hingga 150 epoch dan checkpoint


In [8]:
final_path = OUT / "model" / "cnn.keras"
checkpoint = OUT / "model" / "best_checkpoint.keras"
report_path = OUT / "laporan" / "cnn_training.json"
if final_path.exists() and report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    if (
        report.get("identity_sha256") != identity_sha
        or report.get("cnn_sha256") != sha256_file(final_path)
        or report.get("selected") != best
    ):
        raise ValueError("CNN v5 tersimpan tidak sesuai ledger")
    print("CNN yang cocok sudah ada:", final_path)
elif final_path.exists() or report_path.exists():
    raise FileExistsError("Artefak CNN hanya sebagian ada; periksa sebelum melanjutkan")
else:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = compile_cnn(build_cnn(VARIANT), best["lr"], 150, 1250)
    progress_path = OUT / "laporan" / "cnn_training_progress.json"
    if progress_path.exists():
        previous_progress = json.loads(progress_path.read_text(encoding="utf-8"))
        if previous_progress.get("identity_sha256") != identity_sha:
            raise ValueError("Progress CNN berasal dari konfigurasi berbeda")
        previous_best = float(previous_progress["best_validation_accuracy"])
    else:
        previous_best = None

    class SaveBestHistory(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            value = float((logs or {})["val_accuracy"])
            best_value = max(value, previous_best or 0.0)
            if progress_path.exists():
                saved = json.loads(progress_path.read_text(encoding="utf-8"))
                best_value = max(best_value, float(saved["best_validation_accuracy"]))
            atomic_json(
                progress_path,
                {
                    "identity_sha256": identity_sha,
                    "last_epoch": int(epoch) + 1,
                    "best_validation_accuracy": best_value,
                },
            )

    callbacks = [
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(OUT / "model" / "training_backup")
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(checkpoint),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
            initial_value_threshold=previous_best,
        ),
        SaveBestHistory(),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=30,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]
    dataset = make_training_dataset(
        x_all[idx_train], y_all[idx_train], best["policy"], 32, SEED
    )
    x_val = prepare_images(x_all[idx_val], VARIANT)
    y_val = tf.one_hot(y_all[idx_val], 10, dtype=tf.float32)
    began = time.perf_counter()
    history = model.fit(
        dataset,
        validation_data=(x_val, y_val),
        epochs=150,
        callbacks=callbacks,
        verbose=1,
    )
    seconds = time.perf_counter() - began
    if not checkpoint.exists():
        raise FileNotFoundError("Checkpoint CNN terbaik tidak ada")
    selected_model = tf.keras.models.load_model(checkpoint, compile=False)
    selected_model.save(final_path)
    report = {
        "run_id": RUN_ID,
        "variant": VARIANT,
        "seed": SEED,
        "identity_sha256": identity_sha,
        "split_hash": split_hash,
        "selected": best,
        "epochs_run": int(json.loads(progress_path.read_text())["last_epoch"]),
        "seconds": seconds,
        "best_validation_accuracy": float(
            json.loads(progress_path.read_text())["best_validation_accuracy"]
        ),
        "cnn_sha256": sha256_file(final_path),
        "feature_layer": "embedding",
        "feature_dim": 512,
        "post_exposure_test": True,
    }
    atomic_json(report_path, report)
    del model, selected_model, dataset
    tf.keras.backend.clear_session()
    gc.collect()
print(
    "CNN hash:",
    report["cnn_sha256"][:16],
    "| Validation CNN:",
    f"{report['best_validation_accuracy']:.2%}",
)


/home/saga/anaconda3/envs/envGPU/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(



Epoch 1: val_accuracy improved from None to 0.44210, saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/outputs/cifar10_7fitur_v5/avg/model/best_checkpoint.keras



Epoch 1: finished saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/outputs/cifar10_7fitur_v5/avg/model/best_checkpoint.keras


Epoch 1/150 — 1250/1250 ━━━━━━━━━━━━━━━━━━━━ 120s 90ms/step - accuracy: 0.4116 - loss: 1.7430 - val_accuracy: 0.4421 - val_loss: 1.6816



Epoch 2: val_accuracy did not improve from 0.44210


...

Epoch 149/150 — 1250/1250 ━━━━━━━━━━━━━━━━━━━━ 92s 74ms/step - accuracy: 0.9991 - loss: 0.2887 - val_accuracy: 0.9294 - val_loss: 0.5630



Epoch 150: val_accuracy did not improve from 0.92940


Restoring model weights from the end of the best epoch: 149.


CNN hash: ace6426464400461 | Validation CNN: 92.94%


Epoch 150/150 — 1250/1250 ━━━━━━━━━━━━━━━━━━━━ 92s 74ms/step - accuracy: 0.9992 - loss: 0.2885 - val_accuracy: 0.9285 - val_loss: 0.5636


## Akurasi CNN terpilih pada training bersih dan validation


In [9]:
cnn_accuracy_path = OUT / "laporan" / "cnn_accuracy.json"
if cnn_accuracy_path.exists():
    cnn_accuracy = json.loads(cnn_accuracy_path.read_text(encoding="utf-8"))
    if (
        cnn_accuracy.get("cnn_sha256") != report["cnn_sha256"]
        or cnn_accuracy.get("split_hash") != split_hash
    ):
        raise ValueError(
            "Akurasi CNN tersimpan berasal dari checkpoint atau split lain"
        )
else:
    evaluation_model = tf.keras.models.load_model(final_path, compile=False)

    def count_clean_predictions(indices, label):
        correct = 0
        for begin in range(0, len(indices), 128):
            batch_indices = indices[begin : begin + 128]
            batch = prepare_images(x_all[batch_indices], VARIANT)
            scores = evaluation_model(batch, training=False).numpy()
            correct += int(np.sum(np.argmax(scores, axis=1) == y_all[batch_indices]))
            if (begin + len(batch_indices)) % 6400 == 0 or begin + len(
                batch_indices
            ) == len(indices):
                print(label, begin + len(batch_indices), "/", len(indices), flush=True)
        return correct

    train_correct = count_clean_predictions(idx_train, "CNN training bersih")
    val_correct = count_clean_predictions(idx_val, "CNN validation")
    cnn_accuracy = {
        "run_id": RUN_ID,
        "variant": VARIANT,
        "cnn_sha256": report["cnn_sha256"],
        "split_hash": split_hash,
        "train_count": len(idx_train),
        "val_count": len(idx_val),
        "train_correct_clean": train_correct,
        "val_correct": val_correct,
        "train_accuracy_clean": train_correct / len(idx_train),
        "validation_accuracy_clean": val_correct / len(idx_val),
        "note": "Akurasi training dihitung pada gambar asli tanpa augmentasi.",
    }
    atomic_json(cnn_accuracy_path, cnn_accuracy)
    del evaluation_model
    tf.keras.backend.clear_session()
    gc.collect()
print(
    "CNN | training bersih:",
    f"{cnn_accuracy['train_accuracy_clean']:.2%}",
    "| validation:",
    f"{cnn_accuracy['validation_accuracy_clean']:.2%}",
)


W0000 00:00:1791596668.432971   71613 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.63GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


CNN training bersih 6400 / 40000


CNN training bersih 12800 / 40000


CNN training bersih 19200 / 40000


CNN training bersih 25600 / 40000


...

CNN training bersih 38400 / 40000


CNN training bersih 40000 / 40000


CNN validation 6400 / 10000


CNN validation 10000 / 10000


CNN | training bersih: 100.00% | validation: 92.94%


## 9. Ekstraksi embedding sementara untuk melatih SVM


In [10]:
model = tf.keras.models.load_model(final_path, compile=False)
extractor = tf.keras.Model(model.input, model.get_layer("embedding").output)
features = np.empty((50000, 512), np.float32)
began = time.perf_counter()
for begin in range(0, 50000, 128):
    end = min(begin + 128, 50000)
    batch = prepare_images(x_all[begin:end], VARIANT)
    original = extractor(batch, training=False).numpy().astype(np.float32)
    flipped = (
        extractor(np.ascontiguousarray(batch[:, :, ::-1, :]), training=False)
        .numpy()
        .astype(np.float32)
    )
    features[begin:end] = (original + flipped) * np.float32(0.5)
    if end % 6400 == 0 or end == 50000:
        print("Ekstraksi sementara:", end, "/50000", flush=True)
if not np.isfinite(features).all():
    raise ValueError("Embedding CNN mengandung nilai nonfinite")
print(
    "Ekstraksi sementara selesai:",
    features.shape,
    "detik:",
    round(time.perf_counter() - began, 1),
)
del extractor, model
tf.keras.backend.clear_session()
gc.collect()


Ekstraksi sementara: 6400 /50000


Ekstraksi sementara: 12800 /50000


Ekstraksi sementara: 19200 /50000


Ekstraksi sementara: 25600 /50000


Ekstraksi sementara: 32000 /50000


Ekstraksi sementara: 38400 /50000


Ekstraksi sementara: 44800 /50000


Ekstraksi sementara: 50000 /50000


Ekstraksi sementara selesai: (50000, 512) detik: 48.7


0

## 10. GridSearchCV SVM standalone


In [11]:
screen_svm_train, _ = train_test_split(
    idx_train, train_size=12000, stratify=y_all[idx_train], random_state=SEED
)
screen_svm_val, _ = train_test_split(
    idx_val, train_size=3000, stratify=y_all[idx_val], random_state=SEED
)
screen_svm_train, screen_svm_val = np.sort(screen_svm_train), np.sort(screen_svm_val)
search_idx = np.r_[screen_svm_train, screen_svm_val]
fold_svm = PredefinedSplit(
    np.r_[np.full(12000, -1, dtype=np.int8), np.zeros(3000, dtype=np.int8)]
)
svm_grid_path = OUT / "laporan" / "svm_grid.json"
svm_ledger_path = OUT / "laporan" / "svm_grid_ledger.json"
param_grid = {"svc__C": [1.0, 10.0, 30.0], "svc__gamma": ["scale", 0.25, 0.5, 1.0]}


def svm_pipeline(C=1.0, gamma="scale"):
    return Pipeline(
        [
            ("scale", StandardScaler()),
            ("l2", Normalizer()),
            ("svc", SVC(kernel="rbf", C=C, gamma=gamma, cache_size=1024)),
        ]
    )


feature_sha = hashlib.sha256(features.tobytes()).hexdigest()
identity = {
    "run_id": RUN_ID,
    "variant": VARIANT,
    "cnn_sha256": report["cnn_sha256"],
    "split_hash": split_hash,
    "feature_sha256": feature_sha,
    "screen_train_sha256": hashlib.sha256(screen_svm_train.tobytes()).hexdigest(),
    "screen_val_sha256": hashlib.sha256(screen_svm_val.tobytes()).hexdigest(),
    "param_grid": param_grid,
    "pipeline": "standard-l2-rbf-v5",
}
identity_sha = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
if svm_ledger_path.exists():
    svm_ledger = json.loads(svm_ledger_path.read_text(encoding="utf-8"))
    if svm_ledger.get("identity_sha256") != identity_sha:
        raise ValueError("Ledger SVM standalone berasal dari fitur/konfigurasi lain")
else:
    svm_ledger = {"identity_sha256": identity_sha, "identity": identity, "results": {}}
    atomic_json(svm_ledger_path, svm_ledger)
for C in param_grid["svc__C"]:
    for gamma in param_grid["svc__gamma"]:
        key = f"C={C}|gamma={gamma}"
        if key in svm_ledger["results"]:
            print("SVM grid cache:", key, flush=True)
            continue
        grid = GridSearchCV(
            svm_pipeline(),
            {"svc__C": [C], "svc__gamma": [gamma]},
            cv=fold_svm,
            scoring="accuracy",
            refit=False,
            n_jobs=1,
            verbose=1,
            error_score="raise",
        )
        began = time.perf_counter()
        grid.fit(features[search_idx], y_all[search_idx])
        svm_ledger["results"][key] = {
            "C": C,
            "gamma": gamma,
            "accuracy": float(grid.best_score_),
            "seconds": time.perf_counter() - began,
        }
        atomic_json(svm_ledger_path, svm_ledger)
        mean_seconds = np.mean(
            [item["seconds"] for item in svm_ledger["results"].values()]
        )
        remaining = 12 - len(svm_ledger["results"])
        print(
            "SVM grid:",
            key,
            f"{grid.best_score_:.2%}",
            "| perkiraan sisa menit:",
            round(remaining * mean_seconds / 60, 1),
            flush=True,
        )
if len(svm_ledger["results"]) != 12:
    raise RuntimeError("Grid SVM standalone belum lengkap")
winner = max(svm_ledger["results"].values(), key=lambda item: item["accuracy"])
svm_grid_report = {
    "identity_sha256": identity_sha,
    "cnn_sha256": report["cnn_sha256"],
    "split_hash": split_hash,
    "feature_sha256": feature_sha,
    "param_grid": param_grid,
    "best_params": {"svc__C": winner["C"], "svc__gamma": winner["gamma"]},
    "best_accuracy": winner["accuracy"],
    "candidate_results": svm_ledger["results"],
}
if svm_grid_path.exists():
    previous = json.loads(svm_grid_path.read_text(encoding="utf-8"))
    if previous != svm_grid_report:
        raise ValueError("Laporan grid SVM lama berbeda")
else:
    atomic_json(svm_grid_path, svm_grid_report)
print(
    "SVM GridSearchCV terbaik:",
    svm_grid_report["best_params"],
    "accuracy:",
    svm_grid_report["best_accuracy"],
)


Fitting 1 folds for each of 1 candidates, totalling 1 fits


SVM grid: C=1.0|gamma=scale 93.13% | perkiraan sisa menit: 0.2


Fitting 1 folds for each of 1 candidates, totalling 1 fits


SVM grid: C=1.0|gamma=0.25 93.37% | perkiraan sisa menit: 0.1


Fitting 1 folds for each of 1 candidates, totalling 1 fits


...

Fitting 1 folds for each of 1 candidates, totalling 1 fits


SVM grid: C=30.0|gamma=0.5 93.17% | perkiraan sisa menit: 0.0


Fitting 1 folds for each of 1 candidates, totalling 1 fits


SVM grid: C=30.0|gamma=1.0 93.13% | perkiraan sisa menit: 0.0


SVM GridSearchCV terbaik: {'svc__C': 1.0, 'svc__gamma': 0.25} accuracy: 0.9336666666666666


## 11. Validasi SVM pada development 40K/10K


In [12]:
params = svm_grid_report["best_params"]
dev_path = OUT / "model" / "svm_development_40k.joblib"
dev_report_path = OUT / "laporan" / "svm_development.json"
if dev_path.exists() and dev_report_path.exists():
    dev_report = json.loads(dev_report_path.read_text(encoding="utf-8"))
    if (
        dev_report.get("cnn_sha256") != report["cnn_sha256"]
        or dev_report.get("best_params") != params
    ):
        raise ValueError("SVM development lama tidak cocok")
else:
    if dev_path.exists() or dev_report_path.exists():
        raise FileExistsError("SVM development hanya sebagian tersedia")
    pipeline = svm_pipeline(params["svc__C"], params["svc__gamma"])
    began = time.perf_counter()
    pipeline.fit(features[idx_train], y_all[idx_train])
    pred = pipeline.predict(features[idx_val])
    dev_report = {
        "run_id": RUN_ID,
        "variant": VARIANT,
        "cnn_sha256": report["cnn_sha256"],
        "split_hash": split_hash,
        "best_params": params,
        "validation_accuracy": float(accuracy_score(y_all[idx_val], pred)),
        "validation_macro_f1": float(f1_score(y_all[idx_val], pred, average="macro")),
        "seconds": time.perf_counter() - began,
    }
    joblib.dump({"pipeline": pipeline, **dev_report}, dev_path)
    atomic_json(dev_report_path, dev_report)
print("SVM development:", f"{dev_report['validation_accuracy']:.2%}")


SVM development: 93.45%


## Akurasi SVM development pada training dan validation


In [13]:
svm_dev_accuracy_path = OUT / "laporan" / "svm_development_accuracy.json"
if svm_dev_accuracy_path.exists():
    svm_dev_accuracy = json.loads(svm_dev_accuracy_path.read_text(encoding="utf-8"))
    if (
        svm_dev_accuracy.get("cnn_sha256") != report["cnn_sha256"]
        or svm_dev_accuracy.get("best_params") != params
        or svm_dev_accuracy.get("split_hash") != split_hash
    ):
        raise ValueError("Akurasi SVM development tersimpan tidak kompatibel")
else:
    development_model = joblib.load(dev_path)
    if (
        development_model.get("cnn_sha256") != report["cnn_sha256"]
        or development_model.get("best_params") != params
    ):
        raise ValueError("Model SVM development tidak cocok")
    train_predictions = development_model["pipeline"].predict(features[idx_train])
    svm_dev_accuracy = {
        "run_id": RUN_ID,
        "variant": VARIANT,
        "cnn_sha256": report["cnn_sha256"],
        "split_hash": split_hash,
        "best_params": params,
        "train_count": len(idx_train),
        "val_count": len(idx_val),
        "train_correct": int(np.sum(train_predictions == y_all[idx_train])),
        "train_accuracy": float(accuracy_score(y_all[idx_train], train_predictions)),
        "validation_accuracy": float(dev_report["validation_accuracy"]),
    }
    atomic_json(svm_dev_accuracy_path, svm_dev_accuracy)
    del development_model
print(
    "SVM development | training:",
    f"{svm_dev_accuracy['train_accuracy']:.2%}",
    "| validation:",
    f"{svm_dev_accuracy['validation_accuracy']:.2%}",
)


SVM development | training: 100.00% | validation: 93.45%


## 12. Fit SVM standalone final pada 50K


In [14]:
svm_path = OUT / "model" / "svm.joblib"
svm_report_path = OUT / "laporan" / "svm_training.json"
if svm_path.exists() and svm_report_path.exists():
    svm_report = json.loads(svm_report_path.read_text(encoding="utf-8"))
    if (
        svm_report.get("cnn_sha256") != report["cnn_sha256"]
        or svm_report.get("best_params") != params
    ):
        raise ValueError("SVM final lama tidak cocok")
else:
    if svm_path.exists() or svm_report_path.exists():
        raise FileExistsError("SVM final hanya sebagian tersedia")
    pipeline = svm_pipeline(params["svc__C"], params["svc__gamma"])
    began = time.perf_counter()
    pipeline.fit(features, y_all)
    svm_report = {
        "run_id": RUN_ID,
        "variant": VARIANT,
        "cnn_sha256": report["cnn_sha256"],
        "split_hash": split_hash,
        "best_params": params,
        "fit_count": 50000,
        "representation": "B",
        "feature_layer": "embedding",
        "seconds": time.perf_counter() - began,
    }
    joblib.dump({"pipeline": pipeline, **svm_report}, svm_path)
    atomic_json(svm_report_path, svm_report)
print("SVM final 50K:", svm_path)


SVM final 50K: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/outputs/cifar10_7fitur_v5/avg/model/svm.joblib


## Akurasi training SVM final 50K


In [15]:
svm_final_accuracy_path = OUT / "laporan" / "svm_final_train_accuracy.json"
if svm_final_accuracy_path.exists():
    svm_final_accuracy = json.loads(svm_final_accuracy_path.read_text(encoding="utf-8"))
    if (
        svm_final_accuracy.get("svm_sha256") != sha256_file(svm_path)
        or svm_final_accuracy.get("cnn_sha256") != report["cnn_sha256"]
    ):
        raise ValueError("Akurasi SVM final tersimpan berasal dari model lain")
else:
    final_model = joblib.load(svm_path)
    if (
        final_model.get("cnn_sha256") != report["cnn_sha256"]
        or final_model.get("fit_count") != 50000
    ):
        raise ValueError("SVM final tidak cocok dengan fitur training")
    final_train_predictions = final_model["pipeline"].predict(features)
    svm_final_accuracy = {
        "run_id": RUN_ID,
        "variant": VARIANT,
        "cnn_sha256": report["cnn_sha256"],
        "svm_sha256": sha256_file(svm_path),
        "train_count": len(y_all),
        "train_correct": int(np.sum(final_train_predictions == y_all)),
        "train_accuracy": float(accuracy_score(y_all, final_train_predictions)),
    }
    atomic_json(svm_final_accuracy_path, svm_final_accuracy)
    del final_model
print("SVM final | training 50K:", f"{svm_final_accuracy['train_accuracy']:.2%}")


SVM final | training 50K: 98.79%


## Tabel ringkas akurasi model pada tahap Training


In [16]:
rows = [
    (
        "CNN custom",
        cnn_accuracy["train_accuracy_clean"],
        cnn_accuracy["validation_accuracy_clean"],
    ),
    (
        "CNN + SVM development",
        svm_dev_accuracy["train_accuracy"],
        svm_dev_accuracy["validation_accuracy"],
    ),
]
print(f"{'Model':<25} {'Training':>12} {'Validation':>12}")
print("-" * 51)
for name, train_value, val_value in rows:
    print(f"{name:<25} {train_value:>11.2%} {val_value:>11.2%}")
print("CNN + SVM final | training 50K:", f"{svm_final_accuracy['train_accuracy']:.2%}")
print(
    "Training final 50K tidak mempunyai validation terpisah; angka validation di atas berasal dari model development 40K."
)


Model                         Training   Validation
---------------------------------------------------
CNN custom                    100.00%      92.94%
CNN + SVM development         100.00%      93.45%
CNN + SVM final | training 50K: 98.79%
Training final 50K tidak mempunyai validation terpisah; angka validation di atas berasal dari model development 40K.
